# Bank Customer Churn SHAP Analysis

This notebook loads the trained churn model and uses SHAP to explain its predictions. It includes global and local feature importance analysis through SHAP summary, bar, and waterfall plots.

## 1. Import Libraries

We import the libraries needed to load the model, prepare the data, and generate SHAP explanations.

In [1]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import shap
except ImportError:
    shap = None

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Saved Model

The trained model is loaded from the models folder.

In [2]:
model_path = Path("../models/churn_model.pkl")
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

model = joblib.load(model_path)
print("Model loaded successfully from:", model_path)

FileNotFoundError: Model not found at: ..\models\churn_model.pkl

## 3. Load the Preprocessed Dataset

The notebook uses the processed training data as the reference set for SHAP analysis.

In [ ]:
processed_dir = Path("../datasets/processed")
X_train = pd.read_csv(processed_dir / "X_train_scaled.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").iloc[:, 0]

print("Loaded training features shape:", X_train.shape)
print("Loaded training target shape:", y_train.shape)
X_train.head()

## 4. Generate SHAP Explanations

SHAP values are computed for the model using the training data. If SHAP is not available, the notebook explains that the dependency is missing.

In [ ]:
if shap is None:
    print("SHAP is not installed. Please install it to run the explanation workflow.")
else:
    # Use a background sample for faster explanation computation
    background_sample = X_train.sample(n=min(100, len(X_train)), random_state=42)

    # Create a SHAP explainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(background_sample)

    print("SHAP values generated successfully.")
    print("SHAP values type:", type(shap_values))

## 5. Global Feature Importance

Global feature importance shows which variables contribute most to the model's predictions across the dataset.

In [ ]:
if shap is None:
    print("SHAP is not available.")
else:
    # For tree models, SHAP values may be a list or an array depending on the model type
    if isinstance(shap_values, list):
        shap_values_for_plot = shap_values[1]
    else:
        shap_values_for_plot = shap_values

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values_for_plot, background_sample, plot_type="bar", show=False)
    plt.title("Global Feature Importance (SHAP Bar Plot)")
    plt.tight_layout()
    plt.show()

## 6. SHAP Summary Plot

A summary plot combines feature importance and feature effect direction for the model.

In [ ]:
if shap is None:
    print("SHAP is not available.")
else:
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values_for_plot, background_sample, show=False)
    plt.title("SHAP Summary Plot")
    plt.tight_layout()
    plt.show()

## 7. Local Feature Importance

Local explanations help us understand why the model made a specific prediction for an individual sample.

In [ ]:
if shap is None:
    print("SHAP is not available.")
else:
    sample = X_train.iloc[[0]].copy()
    single_shap = explainer.shap_values(sample)

    if isinstance(single_shap, list):
        single_shap_for_plot = single_shap[1]
    else:
        single_shap_for_plot = single_shap

    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        single_shap_for_plot[0],
        feature_names=sample.columns.tolist(),
        show=False
    )
    plt.title("SHAP Waterfall Plot for a Single Prediction")
    plt.tight_layout()
    plt.show()

## 8. Interpretation Summary

The final section summarizes what these SHAP analyses reveal about the model's decision-making behavior.

In [ ]:
print("SHAP analysis completed.")
print("Interpretation guidance:")
print("- Global plots help identify the most influential features across the dataset.")
print("- Local waterfall plots explain the model's reasoning for one prediction.")
print("- These explanations support model transparency and stakeholder communication.")